# Premier League Probability Engine

## Phase 2: Feature Engineering

The Elo model developed in Notebook 1 provides a baseline measure of team strength.

This notebook focuses on constructing the features that will be available before each match. These features will later be used to predict match outcomes and compare the model's probabilities with bookmaker odds.

The key principle throughout this notebook is to avoid look-ahead bias. Every feature must be calculated using only information that would have been known before kickoff.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

## 1. Loading multiple Premier League seasons

The first Elo model used only the 2024–25 season.

For predictive modelling, several seasons are needed so that team ratings and rolling features have enough historical information. This also means teams do not all begin the target season with identical ratings.

The code below loads Premier League match data from 2015–16 through 2024–25 and combines the seasons into one chronological dataset.

In [2]:
season_codes = {
    "2015-16": "1516",
    "2016-17": "1617",
    "2017-18": "1718",
    "2018-19": "1819",
    "2019-20": "1920",
    "2020-21": "2021",
    "2021-22": "2122",
    "2022-23": "2223",
    "2023-24": "2324",
    "2024-25": "2425",
}

season_frames = []

for season_name, season_code in season_codes.items():
    url = (
        "https://www.football-data.co.uk/mmz4281/"
        f"{season_code}/E0.csv"
    )

    season_data = pd.read_csv(url)

    season_data["Season"] = season_name

    season_frames.append(season_data)

matches_all = pd.concat(
    season_frames,
    ignore_index=True,
)

print(f"Seasons loaded: {len(season_frames)}")
print(f"Total matches: {len(matches_all)}")

matches_all[["Season", "Date", "HomeTeam", "AwayTeam"]].head()

Seasons loaded: 10
Total matches: 3800


C:\Users\kiera\AppData\Local\Temp\ipykernel_6844\2526682352.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  season_data["Season"] = season_name


,Season,Date,HomeTeam,AwayTeam
0,2015-16,08/08/2015,Bournemouth,Aston Villa
1,2015-16,08/08/2015,Chelsea,Swansea
2,2015-16,08/08/2015,Everton,Watford
3,2015-16,08/08/2015,Leicester,Sunderland
4,2015-16,08/08/2015,Man United,Tottenham


## 2. Checking column consistency

Before combining data from different seasons, it is important to confirm that the files use compatible column names.

Football-data files sometimes gain or lose bookmaker columns between seasons. The core match columns should remain consistent, but this must be checked rather than assumed.

In [3]:
column_counts = {}

for season_name, season_code in season_codes.items():
    url = (
        "https://www.football-data.co.uk/mmz4281/"
        f"{season_code}/E0.csv"
    )

    season_data = pd.read_csv(url)

    column_counts[season_name] = len(season_data.columns)

pd.Series(column_counts, name="Number of columns")

2015-16     65
2016-17     65
2017-18     65
2018-19     62
2019-20    106
2020-21    106
2021-22    106
2022-23    106
2023-24    106
2024-25    120
Name: Number of columns, dtype: int64

In [4]:
required_columns = [
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
]

missing_columns = {}

for season_name, season_code in season_codes.items():
    url = (
        "https://www.football-data.co.uk/mmz4281/"
        f"{season_code}/E0.csv"
    )

    season_data = pd.read_csv(url)

    missing = [
        column
        for column in required_columns
        if column not in season_data.columns
    ]

    missing_columns[season_name] = missing

missing_columns

{'2015-16': [],
 '2016-17': [],
 '2017-18': [],
 '2018-19': [],
 '2019-20': [],
 '2020-21': [],
 '2021-22': [],
 '2022-23': [],
 '2023-24': [],
 '2024-25': []}

## 3. Selecting the core match columns

The raw files contain many bookmaker and match-statistics columns that vary between seasons.

For the first feature-engineering stage, only the columns needed to identify each match and its final result are retained. Additional variables can be added later when they have a clear modelling purpose.

In [5]:
core_columns = [
    "Season",
    "Date",
    "HomeTeam",
    "AwayTeam",
    "FTHG",
    "FTAG",
    "FTR",
]

matches = matches_all[core_columns].copy()

matches.head()

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,2015-16,08/08/2015,Bournemouth,Aston Villa,0,1,A
1,2015-16,08/08/2015,Chelsea,Swansea,2,2,D
2,2015-16,08/08/2015,Everton,Watford,2,2,D
3,2015-16,08/08/2015,Leicester,Sunderland,4,2,H
4,2015-16,08/08/2015,Man United,Tottenham,1,0,H


In [6]:
print(f"Rows: {matches.shape[0]}")
print(f"Columns: {matches.shape[1]}")

Rows: 3800
Columns: 7


In [7]:
matches.isna().sum()

Season      0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
dtype: int64

## 4. Converting dates and ordering matches

All predictive features must be calculated using information available before each match.

The matches are therefore converted into proper datetime values and sorted chronologically. This ensures that rolling statistics and Elo ratings are updated in the correct order and prevents look-ahead bias.

In [8]:
matches["Date"] = pd.to_datetime(
    matches["Date"],
    dayfirst=True,
    errors="coerce",
)

matches["Date"].isna().sum()

np.int64(380)

In [9]:
matches = (
    matches
    .sort_values(["Date", "HomeTeam", "AwayTeam"])
    .reset_index(drop=True)
)

matches[["Season", "Date", "HomeTeam", "AwayTeam"]].head(10)

,Season,Date,HomeTeam,AwayTeam
0,2015-16,2015-08-08,Bournemouth,Aston Villa
1,2015-16,2015-08-08,Chelsea,Swansea
2,2015-16,2015-08-08,Everton,Watford
3,2015-16,2015-08-08,Leicester,Sunderland
4,2015-16,2015-08-08,Man United,Tottenham
5,2015-16,2015-08-08,Norwich,Crystal Palace
6,2015-16,2015-08-09,Arsenal,West Ham
7,2015-16,2015-08-09,Newcastle,Southampton
8,2015-16,2015-08-09,Stoke,Liverpool
9,2015-16,2015-08-10,West Brom,Man City


In [10]:
print("First match date:", matches["Date"].min())
print("Last match date:", matches["Date"].max())

First match date: 2015-08-08 00:00:00
Last match date: 2025-05-25 00:00:00
